In [0]:
%python
bronze_path = '/Volumes/workspace/techvenda/filestore/bronze/'
silver_path = '/Volumes/workspace/techvenda/filestore/silver/'
gold_path = '/Volumes/workspace/techvenda/filestore/gold/'
origem_path = '/Volumes/workspace/techvenda/filestore/origem/'

In [0]:
%python
#Tabelas temporarias
silver_mapeamento= {
    'temp_silver_clientes' : f'{silver_path}/clientes/',
    'temp_silver_itens_pedido' : f'{silver_path}/itens_pedido/',
    'temp_silver_pedidos' : f'{silver_path}/pedidos/',
    'temp_silver_produtos' : f'{silver_path}/produtos/',
    'temp_silver_vendedores' : f'{silver_path}/vendedores/'

}
for view_name, path in silver_mapeamento.items():
    (spark.read.format('delta')
        .load(path)
        .createOrReplaceTempView(view_name)
    )

In [0]:
%sql
select * from temp_silver_pedidos

In [0]:
%sql
select * from temp_silver_clientes

In [0]:
%sql
select * from temp_silver_itens_pedido

In [0]:
%sql
select * from temp_silver_produtos

In [0]:
%python
#criando e tratando tabela faturamento mensal conform regra de negocio e salvando na Gold
faturamento_mensal = spark.sql("""
SELECT
    YEAR(TO_DATE(p.dataPedido, 'dd/MM/yyyy')) AS ano,
    MONTH(TO_DATE(p.dataPedido, 'dd/MM/yyyy')) as mes,
  

    COUNT(DISTINCT p.id_pedido) AS total_pedidos,

    ROUND(SUM(
        pr.preco_unitario * ip.quantidade), 2
    ) AS receita_total

FROM temp_silver_pedidos p

LEFT JOIN temp_silver_itens_pedido ip
    ON p.id_pedido = ip.id_pedido
LEFT JOIN temp_silver_produtos pr
    ON ip.id_produto = pr.id_produto

GROUP BY
    YEAR(to_date(p.dataPedido, 'dd/MM/yyyy')),
    MONTH(to_date(p.dataPedido, 'dd/MM/yyyy')),
    DAY(to_date(p.dataPedido, 'dd/MM/yyyy'))

ORDER BY
    ano ASC,
    mes ASC

""")

# salvar em Delta na Gold
faturamento_mensal.write\
    .mode('overwrite')\
        .format('delta')\
            .option('mergeShema','true')\
                .save(f'{gold_path}/faturamento_mensal/')

In [0]:
%sql
--Criando tabela fisica
create table if not exists workspace.techvenda.faturamento_mensal 
select * from delta. `/Volumes/workspace/techvenda/filestore/gold/faturamento_mensal/`